# Flickr30k Student Baseline Audit

Notebook CPU này kiểm định toàn vẹn và tổng hợp **9 baseline Student**:
S1, S2, S3 × seed 42, 123, 2026. Notebook chỉ đọc output Validation,
không tìm hoặc mở tập Test.

## Kaggle inputs bắt buộc — đúng 9 input

1. Output `F30K_S1_42`;
2. Output `F30K_S1_123`;
3. Output `F30K_S1_2026`;
4. Output `F30K_S2_42`;
5. Output `F30K_S2_123`;
6. Output `F30K_S2_2026`;
7. Output `F30K_S3_42`;
8. Output `F30K_S3_123`;
9. Output `F30K_S3_2026`.

Không gắn dataset ảnh, Data Contract, Base, MS COCO, Teacher, KD,
`pycocoevalcap` hoặc Test caption.

## Runtime

- Accelerator: **None / CPU**;
- Internet: **Off**;
- dự kiến: **2–5 phút**.

## Gate

Notebook xác minh SHA-256, checkpoint, metric, Early Stopping,
1.014 dự đoán Validation mỗi run và trạng thái Test còn niêm phong.
Sau khi gate đạt mới chạy Teacher T1/T2/T3.


In [1]:
"""Audit and aggregate the nine Flickr30k student-baseline runs.

This phase reads Validation artifacts only. The held-out Test split remains
sealed and is neither discovered nor evaluated.
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import statistics
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    import torch
except ImportError:  # Local lightweight audit; Kaggle always provides torch.
    torch = None


INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUTPUT_ROOT = Path(
    os.environ.get(
        "KLCN_OUTPUT_ROOT",
        "/kaggle/working/F30K_S_Audit",
    )
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEEDS = (42, 123, 2026)
MODELS = {
    "S1": "MobileNetV3-Large",
    "S2": "MobileViT-S",
    "S3": "TinyViT-5M",
}
METRICS = ("ce_loss", "BLEU_1", "BLEU_4", "METEOR", "ROUGE_L", "CIDEr_D")
MIN_DELTA = 0.001
MAXIMUM_EPOCHS = 20
MINIMUM_EPOCHS = 5
PATIENCE = 4
VALIDATION_IMAGES = 1_014


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: Any) -> None:
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def unique_result(experiment_id: str) -> Path:
    filename = f"{experiment_id}_RESULT.json"
    matches = sorted(INPUT_ROOT.rglob(filename))
    if len(matches) != 1:
        raise AssertionError(
            f"Cần đúng một output {experiment_id}; tìm thấy {len(matches)}: "
            f"{matches}"
        )
    return matches[0]


def read_metrics(path: Path) -> list[dict[str, float]]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise AssertionError(f"metrics.csv rỗng: {path}")
    return [
        {
            key: float(value)
            for key, value in row.items()
            if key is not None and value not in (None, "")
        }
        for row in rows
    ]


def protocol_selected_row(
    rows: list[dict[str, float]],
) -> tuple[dict[str, float], int]:
    best = -math.inf
    selected: dict[str, float] | None = None
    stale = 0
    for row in rows:
        improved = row["CIDEr_D"] > best + MIN_DELTA
        if improved:
            best = row["CIDEr_D"]
            selected = row
            stale = 0
        else:
            stale += 1
    if selected is None:
        raise AssertionError("Không chọn được checkpoint Validation")
    return selected, stale


def close(left: float, right: float, tolerance: float = 1e-10) -> bool:
    return math.isclose(float(left), float(right), abs_tol=tolerance, rel_tol=0.0)


detail_rows: list[dict[str, Any]] = []
audit_runs: list[dict[str, Any]] = []

for model_id, model_name in MODELS.items():
    for seed in SEEDS:
        experiment_id = f"F30K_{model_id}_{seed}"
        result_path = unique_result(experiment_id)
        run_root = result_path.parent
        result = load_json(result_path)

        assert result["schema_version"] == "f30k_student_baseline_result_v1"
        assert result["status"] == "complete"
        assert result["experiment_id"] == experiment_id
        assert result["model_id"] == model_id
        assert result["model_name"] == model_name
        assert int(result["seed"]) == seed
        assert result["selection_metric"] == "Validation CIDEr-D"
        assert result["early_stopping_enabled"] is True
        assert int(result["maximum_epochs"]) == MAXIMUM_EPOCHS
        assert result["test_set_status"] == "sealed"
        assert result["test_set_used"] is False

        artifact_paths: dict[str, Path] = {}
        expected_names = {
            "best.pt",
            "last.pt",
            "metrics.csv",
            "best_validation_captions.json",
        }
        assert set(result["artifacts"]) == expected_names
        for name, metadata in result["artifacts"].items():
            path = run_root / name
            assert path.is_file(), path
            assert path.stat().st_size == int(metadata["bytes"]), name
            assert sha256(path) == metadata["sha256"], name
            artifact_paths[name] = path

        rows = read_metrics(artifact_paths["metrics.csv"])
        epochs = [int(row["epoch"]) for row in rows]
        assert MINIMUM_EPOCHS <= len(rows) <= MAXIMUM_EPOCHS
        assert epochs == list(range(1, len(rows) + 1))
        assert all(int(row["global_step"]) == int(row["epoch"]) * 454 for row in rows)
        assert all(
            math.isfinite(float(row[key]))
            for row in rows
            for key in (*METRICS, "train_ce", "elapsed_hours")
        )

        selected_row, final_stale = protocol_selected_row(rows)
        selected = result["best_validation_metrics"]
        assert int(result["best_epoch"]) == int(selected_row["epoch"])
        for key in (*METRICS, "train_ce", "global_step", "elapsed_hours", "epoch"):
            assert close(float(selected[key]), selected_row[key]), (experiment_id, key)
        if len(rows) < MAXIMUM_EPOCHS:
            assert final_stale >= PATIENCE

        captions = load_json(artifact_paths["best_validation_captions.json"])
        assert set(captions) == {"references", "hypotheses"}
        references = captions["references"]
        hypotheses = captions["hypotheses"]
        assert len(references) == VALIDATION_IMAGES
        assert len(hypotheses) == VALIDATION_IMAGES
        assert set(references) == set(hypotheses)
        assert set(map(len, references.values())) == {5}
        assert all(
            isinstance(values, list)
            and len(values) == 1
            and isinstance(values[0], str)
            and values[0].strip()
            for values in hypotheses.values()
        )

        checkpoint_payload_verified = torch is not None
        if checkpoint_payload_verified:
            best_checkpoint = torch.load(
                artifact_paths["best.pt"],
                map_location="cpu",
                weights_only=False,
            )
            last_checkpoint = torch.load(
                artifact_paths["last.pt"],
                map_location="cpu",
                weights_only=False,
            )
            for checkpoint in (best_checkpoint, last_checkpoint):
                assert checkpoint["experiment_id"] == experiment_id
                assert checkpoint["model_id"] == model_id
                assert int(checkpoint["seed"]) == seed
                assert checkpoint["test_set_status"] == "sealed"
                assert checkpoint["test_set_used"] is False
            assert int(best_checkpoint["epoch"]) + 1 == int(result["best_epoch"])
            assert int(last_checkpoint["epoch"]) + 1 == len(rows)
            assert close(
                best_checkpoint["metrics"]["CIDEr_D"],
                selected["CIDEr_D"],
            )
            del best_checkpoint, last_checkpoint

        detail_row = {
            "model_id": model_id,
            "model_name": model_name,
            "seed": seed,
            "epochs_completed": len(rows),
            "best_epoch": int(result["best_epoch"]),
            **{key: float(selected[key]) for key in METRICS},
            "train_ce": float(selected["train_ce"]),
            "global_step": int(selected["global_step"]),
            "elapsed_hours": float(selected["elapsed_hours"]),
            "result_sha256": sha256(result_path),
        }
        detail_rows.append(detail_row)
        audit_runs.append(
            {
                "experiment_id": experiment_id,
                "status": "passed",
                "artifact_count": len(artifact_paths),
                "validation_images": len(references),
                "epochs_completed": len(rows),
                "best_epoch": int(result["best_epoch"]),
                "best_validation_cider_d": float(selected["CIDEr_D"]),
                "checkpoint_sha256_verified": True,
                "checkpoint_payload_verified": checkpoint_payload_verified,
                "test_set_status": "sealed",
                "test_set_used": False,
            }
        )
        print(
            f"{experiment_id}: PASSED | epochs={len(rows):02d} | "
            f"best={int(result['best_epoch']):02d} | "
            f"CIDEr-D={float(selected['CIDEr_D']):.6f}"
        )

detail_csv = OUTPUT_ROOT / "F30K_STUDENT_THREE_SEED_METRICS.csv"
with detail_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(detail_rows[0]))
    writer.writeheader()
    writer.writerows(detail_rows)

summary_rows: list[dict[str, Any]] = []
for model_id, model_name in MODELS.items():
    model_rows = [row for row in detail_rows if row["model_id"] == model_id]
    summary: dict[str, Any] = {
        "model_id": model_id,
        "model_name": model_name,
        "seed_count": len(model_rows),
    }
    for metric in METRICS:
        values = [float(row[metric]) for row in model_rows]
        summary[f"{metric}_mean"] = statistics.mean(values)
        summary[f"{metric}_sample_sd"] = statistics.stdev(values)
    summary["elapsed_hours_mean"] = statistics.mean(
        float(row["elapsed_hours"]) for row in model_rows
    )
    summary_rows.append(summary)

summary_csv = OUTPUT_ROOT / "F30K_STUDENT_SUMMARY.csv"
with summary_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)

audit_path = OUTPUT_ROOT / "F30K_STUDENT_BASELINE_AUDIT.json"
audit = {
    "schema_version": "f30k_student_baseline_audit_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed",
    "gate": "student_baseline_three_seed_complete",
    "expected_run_count": 9,
    "verified_run_count": len(audit_runs),
    "models": MODELS,
    "seeds": list(SEEDS),
    "selection_metric": "Validation CIDEr-D",
    "minimum_cider_improvement": MIN_DELTA,
    "early_stopping": {
        "minimum_completed_epochs": MINIMUM_EPOCHS,
        "patience": PATIENCE,
        "maximum_epochs": MAXIMUM_EPOCHS,
    },
    "validation_image_count": VALIDATION_IMAGES,
    "runs": audit_runs,
    "summary": summary_rows,
    "test_set_status": "sealed",
    "test_set_used": False,
    "next_phase_authorized": "Flickr30k Teacher T1/T2/T3 evaluation or adaptation",
}
write_json(audit_path, audit)

report_path = OUTPUT_ROOT / "F30K_STUDENT_BASELINE_AUDIT.md"
lines = [
    "# Flickr30k Student baseline audit",
    "",
    "- Gate: **PASSED**.",
    "- Runs: **9/9**.",
    "- Selection: Validation CIDEr-D, `min_delta = 0.001`.",
    "- Held-out Test: **SEALED / NOT USED**.",
    "",
    "| Model | BLEU-1 | BLEU-4 | METEOR | ROUGE-L | CIDEr-D |",
    "|---|---:|---:|---:|---:|---:|",
]
for row in summary_rows:
    lines.append(
        f"| {row['model_id']} {row['model_name']} | "
        f"{row['BLEU_1_mean']:.6f} ± {row['BLEU_1_sample_sd']:.6f} | "
        f"{row['BLEU_4_mean']:.6f} ± {row['BLEU_4_sample_sd']:.6f} | "
        f"{row['METEOR_mean']:.6f} ± {row['METEOR_sample_sd']:.6f} | "
        f"{row['ROUGE_L_mean']:.6f} ± {row['ROUGE_L_sample_sd']:.6f} | "
        f"{row['CIDEr_D_mean']:.6f} ± {row['CIDEr_D_sample_sd']:.6f} |"
    )
lines.extend(
    [
        "",
        "Bước tiếp theo được phép: Teacher T1/T2/T3 trên Flickr30k. "
        "Không mở Test.",
        "",
    ]
)
report_path.write_text("\n".join(lines), encoding="utf-8")

manifest_path = OUTPUT_ROOT / "F30K_STUDENT_AUDIT_MANIFEST.json"
generated = [audit_path, detail_csv, summary_csv, report_path]
write_json(
    manifest_path,
    {
        "schema_version": "f30k_student_audit_manifest_v1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "artifacts": [
            {
                "name": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256(path),
            }
            for path in generated
        ],
    },
)

print("=" * 72)
print("Audit status          : PASSED")
print("Verified runs         : 9 / 9")
print("Validation images     : 1,014 per run")
print("Held-out Test         : SEALED / NOT USED")
print("Next                  : Flickr30k Teacher T1/T2/T3")
print("Output                :", OUTPUT_ROOT)


F30K_S1_42: PASSED | epochs=14 | best=10 | CIDEr-D=0.473149
F30K_S1_123: PASSED | epochs=20 | best=20 | CIDEr-D=0.476197
F30K_S1_2026: PASSED | epochs=20 | best=19 | CIDEr-D=0.476958
F30K_S2_42: PASSED | epochs=20 | best=19 | CIDEr-D=0.501032
F30K_S2_123: PASSED | epochs=20 | best=19 | CIDEr-D=0.485284
F30K_S2_2026: PASSED | epochs=20 | best=18 | CIDEr-D=0.484101
F30K_S3_42: PASSED | epochs=20 | best=20 | CIDEr-D=0.570692
F30K_S3_123: PASSED | epochs=17 | best=13 | CIDEr-D=0.554664
F30K_S3_2026: PASSED | epochs=20 | best=18 | CIDEr-D=0.576835
Audit status          : PASSED
Verified runs         : 9 / 9
Validation images     : 1,014 per run
Held-out Test         : SEALED / NOT USED
Next                  : Flickr30k Teacher T1/T2/T3
Output                : /kaggle/working/F30K_S_Audit
